# Best-of-N Speed Benchmark Across Quantization Levels

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple quantization settings, such as fp16 and
GPTQ-int4 model variants.

Each configuration is loaded with vLLM, warmed up, timed for
num_trials runs, and then unloaded before the next configuration is
tested. This keeps the comparison focused on how quantization affects
throughput under the same benchmark settings.

Use this notebook to compare speed across precision or quantization
levels. Use benchmark_speed_bon_models_v1.ipynb for the separate
model-axis sweep at a fixed precision.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

import statistics

from utils.configs import GenConfig

from utils.load_data import load_data_hf
from unittests.notebook_utils import benchmark_bon_speed

In [2]:
# Dataset path
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

In [3]:
# Quantization configs to benchmark.
# GPTQ requires a pre-quantized model directory (point model_dir to it).
def _cfg(name, subdir, quantization, dtype):
    return {
        "name":         name,
        "model_dir":    os.path.join(base_dir, subdir),
        "quantization": quantization,
        "load_format":  "auto",
        "dtype":        dtype,
    }


# Full model list (weight-memory GB from benchmark_llm_mem_sizes_v1 in
# the comments). The two fp16 7B models (~16.9 GB) are commented out —
# too large to co-reside comfortably; their GPTQ-int4 variants stay.
quant_configs = [
    _cfg("llama-1b fp16",       "Llama3.2-1B-Instruct",           None,   "float16"),  # 2.80
    _cfg("llama-3b fp16",       "Llama3.2-3B-Instruct",           None,   "float16"),  # 6.47
    _cfg("llama-3b gptq",       "Llama3.2-3B-Instruct-GPTQ",      "gptq", "auto"),     # 2.57
    _cfg("qwen-3b fp16",        "Qwen2.5-3B-Instruct",            None,   "float16"),  # 8.41
    _cfg("qwen-3b gptq-int4",   "Qwen2.5-3B-Instruct-GPTQ-Int4",  "gptq", "auto"),     # 4.63
    # _cfg("qwen-7b fp16",      "Qwen2.5-7B-Instruct",            None,   "float16"),  # 16.86 — large
    _cfg("qwen-7b gptq-int4",   "Qwen2.5-7B-Instruct-GPTQ-Int4",  "gptq", "auto"),     # 7.83
    _cfg("qwen-math-1.5b fp16", "Qwen2.5-Math-1.5B-Instruct",     None,   "float16"),  # 5.54
    # _cfg("qwen-math-7b fp16", "Qwen2.5-Math-7B-Instruct",       None,   "float16"),  # 16.87 — large
]

In [4]:
# Best-of-N search params.
# Aligned with benchmark_speed_bon_models_v1 so the two notebooks run
# the SAME workload (n, gmu, questions, trials) and their numbers are
# directly comparable. (n was 256 here historically; lowered to 32 to
# match the search's real BoN width and the models notebook.)
config = GenConfig()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs (kept identical across both speed notebooks)
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 5                  # cap on questions per benchmark
num_trials = 2                     # timed runs per config
warmup = 1                         # untimed warmup runs per config
llm_gpu_memory_utilization = 0.3

In [5]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 5


## Run benchmark

One config at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [6]:
results = []
for qcfg in quant_configs:
    name, times = benchmark_bon_speed(
        qcfg, config, batch_of_questions, num_trials,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        warmup=warmup,
    )
    results.append((name, times))


=== llama-1b fp16 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.32s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  trial 0:   25.75s total, 5.1491s/question
  trial 1:   17.47s total, 3.4941s/question


[rank0]:[W619 07:04:10.205236823 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== llama-3b fp16 ===


...

[rank0]:[W619 07:28:45.538778263 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== qwen-math-1.5b fp16 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.51s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.51s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  trial 0:   71.42s total, 14.2842s/question
  trial 1:   70.96s total, 14.1930s/question


[rank0]:[W619 07:32:41.853524905 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## Summary

In [7]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'quantization':<25}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<25}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )

=== Summary (level=4, n_questions=5, n_trials=2) ===
quantization               mean s/trial     std    s/question
-------------------------------------------------------------
llama-1b fp16                     21.61    5.85        4.3216
llama-3b fp16                     63.20    1.93       12.6401
llama-3b gptq                     67.87    2.15       13.5747
qwen-3b fp16                     122.21    2.70       24.4414
qwen-3b gptq-int4                102.38    2.84       20.4752
qwen-7b gptq-int4                 98.09    5.05       19.6189
qwen-math-1.5b fp16               71.19    0.32       14.2386
